# Jacobians: Differential Motion, Velocities and Static Forces

This tutorial follows the **Stanford CS223A** lectures by
**Oussama Khatib & Krasimir Kolarov**, using [MuJoCo](https://mujoco.org/)
and `matplotlib` to make each concept concrete.
We use the same 3‑link planar arm from notebook 01.

## Prerequisites

```bash
pip install mujoco numpy matplotlib ipympl ipywidgets mediapy
```

In [1]:
import os, time, itertools
import numpy as np
import mujoco
import mediapy as media
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa

try:
    get_ipython().run_line_magic('matplotlib', 'widget')
except Exception:
    get_ipython().run_line_magic('matplotlib', 'inline')

np.set_printoptions(precision=3, suppress=True, linewidth=100)
print(f"MuJoCo {mujoco.__version__}  |  NumPy {np.__version__}")

MuJoCo 3.9.0  |  NumPy 2.4.6


---

## 1.  The 3‑Link Planar Arm (Recap)

$L_1 = 0.5$, $L_2 = 0.4$, $L_3 = 0.3$ — three revolute joints about $\hat{Z}$.

Forward kinematics:
$$
\begin{aligned}
x &= L_1 c_1 + L_2 c_{12} + L_3 c_{123} \\
y &= L_1 s_1 + L_2 s_{12} + L_3 s_{123} \\
\phi &= \theta_1 + \theta_2 + \theta_3
\end{aligned}
$$

In [2]:
L1, L2, L3 = 0.5, 0.4, 0.3

arm3_xml = """
<mujoco model="planar_3r">
  <compiler angle="radian"/>
  <option gravity="0 0 0"/>
  <worldbody>
    <light name="top" pos="0 0 2"/>
    <geom name="ground" type="plane" size="1.5 1.5 0.01" rgba="0.9 0.9 0.9 1"/>
    <body name="base" pos="0 0 0">
      <geom name="pedestal" type="cylinder" size="0.05 0.02" rgba="0.3 0.3 0.3 1"/>
      <body name="link1" pos="0 0 0">
        <joint name="j1" type="hinge" axis="0 0 1"/>
        <geom name="l1" type="capsule" fromto="0 0 0 0.5 0 0" size="0.03" rgba="0.2 0.6 1.0 1"/>
        <body name="link2" pos="0.5 0 0">
          <joint name="j2" type="hinge" axis="0 0 1"/>
          <geom name="l2" type="capsule" fromto="0 0 0 0.4 0 0" size="0.025" rgba="1.0 0.4 0.2 1"/>
          <body name="link3" pos="0.4 0 0">
            <joint name="j3" type="hinge" axis="0 0 1"/>
            <geom name="l3" type="capsule" fromto="0 0 0 0.3 0 0" size="0.022" rgba="0.4 0.8 0.4 1"/>
            <body name="ee" pos="0.3 0 0">
              <geom name="ee_s" type="sphere" size="0.035" rgba="0.1 0.8 0.1 1"/>
            </body>
          </body>
        </body>
      </body>
    </body>
  </worldbody>
</mujoco>
"""

model = mujoco.MjModel.from_xml_string(arm3_xml)
data   = mujoco.MjData(model)
ee_id  = model.body("ee").id
print(f"nq = {model.nq},  nv = {model.nv},  njnt = {model.njnt}")

nq = 3,  nv = 3,  njnt = 3


In [3]:
# Forward kinematics (analytic)
def fk_analytic(theta1, theta2, theta3):
    t1, t2, t3 = theta1, theta2, theta3
    x = L1 * np.cos(t1) + L2 * np.cos(t1 + t2) + L3 * np.cos(t1 + t2 + t3)
    y = L1 * np.sin(t1) + L2 * np.sin(t1 + t2) + L3 * np.sin(t1 + t2 + t3)
    return np.array([x, y, 0.0]), t1 + t2 + t3

# Shortcut to set qpos and call mj_forward
def set_q(q):
    data.qpos[:] = q
    mujoco.mj_forward(model, data)

set_q([0.5, 0.8, -0.3])
print(f"MuJoCo EE = {np.round(data.xpos[ee_id], 4)}")
print(f"Analytic  = {np.round(fk_analytic(0.5, 0.8, -0.3)[0], 4)}")

MuJoCo EE = [0.708 0.878 0.   ]
Analytic  = [0.708 0.878 0.   ]


---

## 2.  Differential Motion

The **differential motion vector** $D$ bundles infinitesimal translation $dp$
and infinitesimal rotation $\delta$:

$$D = \begin{bmatrix} dp \\ \delta \end{bmatrix} \in \mathbb{R}^6, \qquad
\delta = k \, d\theta \quad (\text{axis}\times\text{angle})$$

For a **revolute joint** the differential end‑effector motion caused by $\delta q_i$ is:

$$\begin{bmatrix} dp \\ \delta \end{bmatrix} =
\begin{bmatrix} z_{i-1} \times (p_{ee} - p_{i-1}) \\ z_{i-1} \end{bmatrix}
\delta q_i$$

where $z_{i-1}$ is the joint axis in world coordinates and $p_{i-1}$ is
the joint position.  **This is the core formula for the geometric Jacobian.**

### 2.1  Skew‑Symmetric Matrix

The cross product $z \times p$ can be written as $\hat{z}\, p$ where
$\hat{z}$ is the skew‑symmetric matrix:

$$\hat{z} = \begin{bmatrix}
0     & -z_z &  z_y \\
z_z  &  0   & -z_x \\
-z_y &  z_x &  0
\end{bmatrix}$$

Then $z \times p = \hat{z}\, p = -\hat{p}\, z$.

In [4]:
def skew(v):
    return np.array([[    0, -v[2],  v[1]],
                     [ v[2],     0, -v[0]],
                     [-v[1],  v[0],    0]])

# Verify: skew(z) @ p  ==  cross(z, p)
z = np.array([1.0, 0.0, 0.0])
p = np.array([0.2, 0.3, 0.4])
print(f"skew(z) @ p = {skew(z) @ p}")
print(f"cross(z,p) = {np.cross(z, p)}")
print(f"Match: {np.allclose(skew(z) @ p, np.cross(z, p))}")

skew(z) @ p = [ 0.  -0.4  0.3]
cross(z,p) = [ 0.  -0.4  0.3]
Match: True


---

## 3.  Geometric Jacobian — Column‑by‑Column Construction

The geometric Jacobian $J(q) \in \mathbb{R}^{6\times n}$ is built one
column at a time.  For a **revolute** joint $i$:

$$J_i = \begin{bmatrix}
z_{i-1} \times (p_{ee} - p_{i-1}) \\ z_{i-1}
\end{bmatrix}$$

where $z_{i-1}$ is the joint axis in world frame and $p_{i-1}$ is the
joint origin in world frame.  Both are retrieved from MuJoCo's `mj_forward`.

The complete Jacobian:

$$J = \big[ J_1 \;\; J_2 \;\; \cdots \;\; J_n \big]$$

In [5]:
def geometric_jacobian(model, data, ee_body_id):
    """Build the full 6xn geometric Jacobian column by column.

    For each revolute joint j:
        J[:, j] = [ z_j x (p_ee - p_j) ;  z_j ]
    """
    nv = model.nv
    J = np.zeros((6, nv))

    p_ee = data.xpos[ee_body_id]

    for j in range(nv):
        # World-frame joint axis (z-axis of the joint)
        z_j = model.jnt_axis[j].copy()          # axis in joint's local frame
        # Transform to world frame
        jnt_body_id = model.jnt_bodyid[j]
        R_world_jnt = data.xmat[jnt_body_id].reshape(3, 3)
        z_world = R_world_jnt @ z_j
        z_world = z_world / np.linalg.norm(z_world)

        # Joint position in world frame
        p_j = data.xanchor[j] if model.jnt_type[j] == mujoco.mjtJoint.mjJNT_HINGE else data.xpos[jnt_body_id]
        # Actually for hinge joints, xanchor gives the joint position

        # Upper 3 rows: z x (p_ee - p_j)
        J[:3, j] = np.cross(z_world, p_ee - p_j)
        # Lower 3 rows: z
        J[3:, j] = z_world

    return J

# Test
q = np.array([0.5, 0.8, -0.3])
set_q(q)
J_geo = geometric_jacobian(model, data, ee_id)
print("Geometric Jacobian:")
print(np.round(J_geo, 4))

Geometric Jacobian:
[[-0.878 -0.638 -0.252]
 [ 0.708  0.269  0.162]
 [ 0.     0.     0.   ]
 [ 0.     0.     0.   ]
 [ 0.     0.     0.   ]
 [ 1.     1.     1.   ]]


### 3.1  Verify Against MuJoCo's `mj_jac`

MuJoCo computes the same Jacobian internally — we simply extract it.

In [6]:
jacp = np.empty((3, model.nv))
jacr = np.empty((3, model.nv))
mujoco.mj_jac(model, data, jacp, jacr, data.xpos[ee_id], ee_id)
J_mj = np.vstack([jacp, jacr])

print("MuJoCo mj_jac:")
print(np.round(J_mj, 4))
print(f"\n||J_geo - J_mj|| = {np.linalg.norm(J_geo - J_mj):.2e}")

MuJoCo mj_jac:
[[-0.878 -0.638 -0.252]
 [ 0.708  0.269  0.162]
 [ 0.     0.     0.   ]
 [ 0.     0.     0.   ]
 [ 0.     0.     0.   ]
 [ 1.     1.     1.   ]]

||J_geo - J_mj|| = 0.00e+00


### 3.2  Analytical Jacobian — From Direct Differentiation

The **analytical Jacobian** $J_A$ is obtained by differentiating the FK
equations directly.  For our 3‑link planar arm the $2\times 3$ position
Jacobian is:

$$
J_A = \begin{bmatrix}
\frac{\partial x}{\partial\theta_1} &
\frac{\partial x}{\partial\theta_2} &
\frac{\partial x}{\partial\theta_3} \\[4pt]
\frac{\partial y}{\partial\theta_1} &
\frac{\partial y}{\partial\theta_2} &
\frac{\partial y}{\partial\theta_3}
\end{bmatrix}
$$

The top two rows of the geometric Jacobian *equal* the analytical Jacobian.

In [7]:
def J_analytic(theta1, theta2, theta3):
    t1, t2, t3 = theta1, theta2, theta3
    s1, s12, s123 = np.sin(t1), np.sin(t1 + t2), np.sin(t1 + t2 + t3)
    c1, c12, c123 = np.cos(t1), np.cos(t1 + t2), np.cos(t1 + t2 + t3)
    return np.array([
        [-L1*s1 - L2*s12 - L3*s123,  -L2*s12 - L3*s123,  -L3*s123],
        [ L1*c1 + L2*c12 + L3*c123,   L2*c12 + L3*c123,   L3*c123],
    ])

J_A = J_analytic(*q)
print("Analytical J (2x3):")
print(np.round(J_A, 4))
print(f"\n||J_A - J_geo[:2]|| = {np.linalg.norm(J_A - J_geo[:2]):.2e}")

Analytical J (2x3):
[[-0.878 -0.638 -0.252]
 [ 0.708  0.269  0.162]]

||J_A - J_geo[:2]|| = 3.10e-16


---

## 4.  Velocity Kinematics

The fundamental relationship:

$$\dot{x} = J(q)\, \dot{q}$$

For a small displacement this becomes $\Delta x \approx J(q)\, \Delta q$.
We verify this numerically and with an animation.

In [8]:
q  = np.array([0.5, 0.8, -0.3])
dq = np.array([0.02, -0.01, 0.015])

ee_a, _ = fk_analytic(*q)
ee_b, _ = fk_analytic(*(q + dq))
dx_numerical = ee_b - ee_a

J = J_analytic(*q)
dx_jacobian = J @ dq

print(f"Δx (FK difference)  = {np.round(dx_numerical, 6)}")
print(f"J·Δq                = {np.round(dx_jacobian, 6)}")
print(f"Error = {np.linalg.norm(dx_numerical - dx_jacobian):.2e}")

Δx (FK difference)  = [-0.015  0.014  0.   ]
J·Δq                = [-0.015  0.014]


ValueError: operands could not be broadcast together with shapes (3,) (2,) 

In [9]:
# Large sweep: error should be O(||dq||²)
errors = []
for _ in range(200):
    dq = np.random.uniform(-0.05, 0.05, 3)
    ee_a, _ = fk_analytic(*q)
    ee_b, _ = fk_analytic(*(q + dq))
    dx = ee_b - ee_a
    err = np.linalg.norm(dx - J @ dq)
    rel_err = err / max(np.linalg.norm(dq), 1e-12)
    errors.append(rel_err)
print(f"Mean relative error: {np.mean(errors):.2e}    (should be << 1)")

ValueError: operands could not be broadcast together with shapes (3,) (2,) 

### 4.1  Resolved Motion Rate Control (Khatib)

Given a desired end‑effector velocity $\dot{x}_d$, we invert the Jacobian
to find the required joint velocities:

$$\dot{q} = J^{-1}(q)\, \dot{x}_d \qquad \text{(non‑redundant)}$$
$$\dot{q} = J^{+}(q)\, \dot{x}_d \qquad \text{(redundant, pseudo‑inverse)}$$

This is **resolved motion rate control (RMRC)** — the core inverse
velocity algorithm.

In [10]:
q = np.array([0.5, 0.8, -0.3])
J = J_analytic(*q)

# Desired EE velocity
dx_desired = np.array([0.05, 0.03])

# Pseudo-inverse solution
J_pinv = np.linalg.pinv(J)
dq_rmrc = J_pinv @ dx_desired

# Verify
dx_achieved = J @ dq_rmrc
print(f"Desired  dx: {dx_desired}")
print(f"RMRC dq:      {np.round(dq_rmrc, 4)}")
print(f"Achieved dx:  {np.round(dx_achieved, 4)}")
print(f"Error: {np.linalg.norm(dx_desired - dx_achieved[:2]):.2e}")

Desired  dx: [0.05 0.03]
RMRC dq:      [ 0.155 -0.283 -0.022]
Achieved dx:  [0.05 0.03]
Error: 6.58e-17


### 4.2  RMRC Animation

The arm follows a desired Cartesian trajectory using resolved motion rate
control.

In [11]:
# Straight-line Cartesian path
start_ee, _ = fk_analytic(0.3, 0.9, -0.5)
end_ee = np.array([0.8, 0.6, 0.0])

dof = 3
nframes, fps = 80, 30
frames = []
opt = mujoco.MjvOption()
cam = mujoco.MjvCamera()
cam.type = mujoco.mjtCamera.mjCAMERA_FREE
cam.distance = 1.8
cam.azimuth = 135
cam.elevation = -30
cam.lookat = [0.5, 0.4, 0]

q_current = np.array([0.3, 0.9, -0.5])
with mujoco.Renderer(model, height=384, width=512) as r:
    for i in range(nframes):
        t = i / (nframes - 1)
        x_desired = start_ee + t * (end_ee - start_ee)
        # RMRC step
        ee_current, _ = fk_analytic(*q_current)
        dq = np.linalg.pinv(J_analytic(*q_current)) @ (x_desired[:2] - ee_current[:2]) * 2.0
        q_current = q_current + dq / fps
        set_q(q_current)
        r.update_scene(data, scene_option=opt, camera=cam)
        frames.append(r.render())
media.show_video(frames, fps=fps)

ee_final, _ = fk_analytic(*q_current)
print(f"Target: {end_ee[:2]}")
print(f"Final : {np.round(ee_final[:2], 4)}")
print(f"Error : {np.linalg.norm(end_ee[:2] - ee_final[:2]):.2e}")

This browser does not support the video tag.


Target: [0.8 0.6]
Final : [0.809 0.62 ]
Error : 2.21e-02


---

## 5.  Singularities — Khatib / Yoshikawa

At a singularity the Jacobian loses rank — the robot cannot generate
end‑effector velocity in some direction regardless of joint rates.

**Yoshikawa's manipulability measure**:

$$w = \sqrt{\det\big(J(q) J^T(q)\big)}$$

When $w \to 0$ the robot is in a singular configuration.  The singular
value decomposition (SVD) of $J$ gives a richer picture:

$$J = U \Sigma V^T, \quad
\Sigma = \mathrm{diag}(\sigma_1, \ldots, \sigma_r)$$

The **condition number** $\kappa = \sigma_{\max} / \sigma_{\min}$ and the
smallest singular value $\sigma_{\min}$ are both indicators of proximity
to a singularity.

In [12]:
def manipulability(Jv):
    return np.float64(np.sqrt(np.abs(np.linalg.det(Jv @ Jv.T))))

def condition_number(Jv):
    return np.linalg.cond(Jv) if np.linalg.norm(Jv) > 1e-12 else np.inf

# Singularity scan for θ3 → 0 (arm fully stretched)
print(f"{'θ3':>8s}  {'w':>10s}  {'κ(Jv)':>10s}  {'σ_min':>10s}")
print("-" * 45)
for t3 in [0.0, 0.005, 0.01, 0.05, 0.2, 0.5, 1.0, np.pi]:
    Jv = J_analytic(0.5, 0.3, t3)
    U, S, Vt = np.linalg.svd(Jv)
    w = manipulability(Jv)
    c = condition_number(Jv)
    marker = " <<<" if S[-1] < 1e-3 else ""
    print(f"{t3:8.3f}  {w:10.2e}  {c:10.2e}  {S[-1]:10.2e}{marker}")

      θ3           w       κ(Jv)       σ_min
---------------------------------------------
   0.000    1.13e-01    1.76e+01    7.99e-02
   0.005    1.14e-01    1.74e+01    8.08e-02
   0.010    1.15e-01    1.72e+01    8.16e-02
   0.050    1.25e-01    1.58e+01    8.87e-02
   0.200    1.64e-01    1.18e+01    1.18e-01
   0.500    2.42e-01    7.53e+00    1.79e-01
   1.000    3.35e-01    4.44e+00    2.74e-01
   3.142    4.67e-02    9.65e+00    6.96e-02


### 5.1  Velocity Ellipsoid (Interactive)

The set of achievable end‑effector velocities for $\|\dot{q}\| = 1$:

$$\{J_v \dot{q} \;:\; \|\dot{q}\| \le 1\}$$

This is an ellipse; its axes are the singular vectors scaled by the
singular values.  Drag the sliders to watch it deform as you approach
a singularity.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

def draw_ellipsoid(ax, Jv, center, scale=0.25, color='#1f77b4', alpha=0.20):
    U, S, _ = np.linalg.svd(Jv)
    angles = np.linspace(0, 2 * np.pi, 200)
    circle = np.vstack([np.cos(angles), np.sin(angles)])
    pts = center[:2, None] + (U @ np.diag(S) @ circle) * scale
    ax.fill(pts[0], pts[1], color=color, alpha=alpha, edgecolor=color, lw=1.5, label='Velocity ellipsoid')
    for j in range(2):
        v = U[:, j] * S[j] * scale
        ax.arrow(center[0], center[1], v[0], v[1], head_width=0.012, head_length=0.012,
                 fc=color, ec=color, lw=1.5)
    return S[-1]

def show_ellipsoid(theta1=0.5, theta2=0.3, theta3=0.5):
    Jv = J_analytic(theta1, theta2, theta3)
    w  = manipulability(Jv)
    c  = condition_number(Jv)
    ee, _ = fk_analytic(theta1, theta2, theta3)

    p0 = np.array([0.0, 0.0])
    p1 = p0 + np.array([L1*np.cos(theta1), L1*np.sin(theta1)])
    p2 = p1 + np.array([L2*np.cos(theta1+theta2), L2*np.sin(theta1+theta2)])
    p3 = p2 + np.array([L3*np.cos(theta1+theta2+theta3), L3*np.sin(theta1+theta2+theta3)])

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot([p0[0], p1[0], p2[0], p3[0]], [p0[1], p1[1], p2[1], p3[1]],
            '-o', lw=3, ms=5, color='#333')
    ax.plot(p3[0], p3[1], 'o', ms=10, color='red', zorder=5, label='EE')

    sigma_min = draw_ellipsoid(ax, Jv, p3[:2], scale=0.35)

    ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
    ax.set_title(fr'  $w={w:.3f}$    $\kappa={c:.2e}$    $\sigma_{{\min}}={sigma_min:.2e}$', fontsize=12)
    ax.set_xlabel('x'); ax.set_ylabel('y'); ax.legend(loc='upper right')
    plt.tight_layout(); plt.show()
    print(f"Manipulability w = {w:.3f}")
    print(f"Condition number \u03ba = {c:.2e}")
    print(f"Smallest singular value \u03c3_min = {sigma_min:.2e}")

_ = widgets.interact(
    show_ellipsoid,
    theta1 = widgets.FloatSlider(min=-np.pi, max=np.pi, step=0.05, value=0.5,  description=r'\u03b8\u2081'),
    theta2 = widgets.FloatSlider(min=-np.pi, max=np.pi, step=0.05, value=0.3,  description=r'\u03b8\u2082'),
    theta3 = widgets.FloatSlider(min=-np.pi, max=np.pi, step=0.05, value=0.5,  description=r'\u03b8\u2083'),
)

### 5.2  Workspace Manipulability Map (Khatib)

Colour each reachable point by $\log_{10} w$ (Yoshikawa measure).
Darker = more dexterous, lighter = near singular.

In [ ]:
theta3_fixed = 0.5
N = 80
t1_vals = np.linspace(-np.pi, np.pi, N)
t2_vals = np.linspace(-np.pi, np.pi, N)
w_map  = np.full((N, N), np.nan)
x_map  = np.full((N, N), np.nan)
y_map  = np.full((N, N), np.nan)

for i, t1 in enumerate(t1_vals):
    for j, t2 in enumerate(t2_vals):
        Jv = J_analytic(t1, t2, theta3_fixed)
        w_map[i, j] = np.log10(max(manipulability(Jv), 1e-12))
        ee, _ = fk_analytic(t1, t2, theta3_fixed)
        x_map[i, j] = ee[0]
        y_map[i, j] = ee[1]

fig, ax = plt.subplots(figsize=(7, 6))
sc = ax.tripcolor(x_map.ravel(), y_map.ravel(), w_map.ravel(),
                  cmap='YlOrRd', shading='gouraud')
plt.colorbar(sc, ax=ax, label=r'$\log_{10} w$')

# Draw a sample arm
tp = (0.5, 0.3)
p0 = [0, 0]
p1 = [L1*np.cos(tp[0]), L1*np.sin(tp[0])]
p2 = [p1[0]+L2*np.cos(tp[0]+tp[1]), p1[1]+L2*np.sin(tp[0]+tp[1])]
ee, _ = fk_analytic(tp[0], tp[1], theta3_fixed)
ax.plot([p0[0], p1[0], p2[0], ee[0]], [p0[1], p1[1], p2[1], ee[1]],
        '-o', lw=2, color='black', ms=4)

ax.set_aspect('equal')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title(fr'Manipulability  $w$  ($\theta_3 = {theta3_fixed:.1f}$)', fontsize=12)
plt.tight_layout(); plt.show()

---

## 6.  Static Forces — The Force Ellipsoid

The relationship between end‑effector force $F$ and joint torques $\tau$
follows from **virtual work**:

$$\tau = J^T(q)\, F$$

This is the **dual** of the velocity relationship.  Where $J$ maps
$\dot{q} \to V$, its transpose maps $F \to \tau$.

The set of achievable end‑effector forces for $\|\tau\| \le 1$ forms
the **force ellipsoid** — its shape is the inverse of the velocity ellipsoid.
Where velocity is poor (near a singularity), force is excellent
(high mechanical advantage).

In [ ]:
theta1, theta2, theta3 = 0.5, 0.8, 1.0
Jv = J_analytic(theta1, theta2, theta3)
U, S, _ = np.linalg.svd(Jv)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5.5))
for ax, title in [(ax1, 'Velocity ellipsoid'), (ax2, 'Force ellipsoid')]:
    ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
    ax.set_xlim(-1.0, 1.0); ax.set_ylim(-1.0, 1.0)
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.set_title(title, fontsize=13)
    ax.axhline(0, color='gray', lw=0.5); ax.axvline(0, color='gray', lw=0.5)

# Velocity ellipsoid (scaled by singular values)
draw_ellipsoid(ax1, Jv, np.zeros(2), scale=0.5, color='#1f77b4', alpha=0.22)

# Force ellipsoid: uses (Jv Jv^T)^{-1/2} = U diag(1/S) as the mapping
Jv_force = U @ np.diag(1.0 / S)  # singular values inverted
draw_ellipsoid(ax2, Jv_force, np.zeros(2), scale=0.5, color='#d62728', alpha=0.22)

fig.suptitle(fr'$\theta = ({theta1:.1f},{theta2:.1f},{theta3:.1f})$  |  '
             fr'$\kappa = {np.linalg.cond(Jv):.2f}$  |  '
             fr'$w = {manipulability(Jv):.3f}$', fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

### 6.1  Verify $\boldsymbol{\tau = J^T F}$ with MuJoCo

Apply an external force to the end‑effector, run inverse dynamics, and
compare the joint torques against $J^T F$.

In [ ]:
q = np.array([0.5, 0.8, -0.3])
set_q(q)
F_ext = np.array([10.0, 5.0, 0.0, 0.0, 0.0, 0.0])  # [fx, fy, fz, mx, my, mz]

data.xfrc_applied[ee_id, :] = F_ext
mujoco.mj_inverse(model, data)
tau_mj = data.qfrc_inverse.copy()[:3]

Jv = J_analytic(*q)
tau_JT = Jv.T @ F_ext[:2]

print(f"F (xy)               = {F_ext[:2]}")
print(f"τ via mj_inverse     = {np.round(tau_mj, 4)}")
print(f"τ = J^T · F          = {np.round(tau_JT, 4)}")
print(f"Error: {np.linalg.norm(tau_mj - tau_JT):.2e}")

---

## 7.  Redundancy — The Null Space of $J$

Our 3‑link arm has 3 joints but only 2 end‑effector position DOFs — it is
**redundant**.  The general solution to $J \dot{q} = \dot{x}$ is:

$$\dot{q} = J^{+}(q)\, \dot{x} + (I - J^{+} J)\, \dot{q}_0$$

The second term projects $\dot{q}_0$ into $\mathcal{N}(J)$ so it produces
**no** end‑effector motion.  We use it for a secondary task.

In [ ]:
q = np.array([0.5, 0.8, -0.3])
Jv = J_analytic(*q)
ee_before, _ = fk_analytic(*q)

J_pinv = np.linalg.pinv(Jv)
N = np.eye(3) - J_pinv @ Jv         # null‑space projector

# Secondary task: pull joint 3 towards -1.0
q_secondary = np.array([0.5, 0.8, -1.0])
dq0 = 0.2 * (q_secondary - q)
dq_null = N @ dq0

print(f"Pre‑projection  dq: {np.round(dq0, 4)}")
print(f"Post‑projection dq: {np.round(dq_null, 4)}")
print(f"Jv · dq    = {np.round(Jv @ dq_null, 6)}  (≈ 0)")

q_new = q + dq_null
ee_after, _ = fk_analytic(*q_new)
print(f"\nEE displacement: {np.linalg.norm(ee_after - ee_before):.2e}")
print(f"θ3 change: {q[2]:.3f} → {q_new[2]:.3f}  (moving toward {q_secondary[2]})")

### 7.1  Null‑Space Motion Animation (Khatib)

The arm reconfigures itself in the null space while the end‑effector
remains stationary — the hallmark of a redundant manipulator.

In [ ]:
q0 = np.array([0.5, 0.8, -0.3])
ee0, _ = fk_analytic(*q0)

nframes, fps = 80, 30
frames = []
opt = mujoco.MjvOption()
cam = mujoco.MjvCamera()
cam.type = mujoco.mjtCamera.mjCAMERA_FREE
cam.distance = 1.8
cam.azimuth = 135
cam.elevation = -30
cam.lookat = [0.5, 0.4, 0]

with mujoco.Renderer(model, height=384, width=512) as r:
    for i in range(nframes):
        Jv = J_analytic(*q0)
        J_pinv = np.linalg.pinv(Jv)
        N = np.eye(3) - J_pinv @ Jv
        # Oscillating secondary goal
        q_sec = np.array([0.5 * np.sin(i * 0.1),
                           0.3 * np.cos(i * 0.13),
                          -0.5 + 0.3 * np.sin(i * 0.08)])
        dq = N @ (0.15 * (q_sec - q0))
        q0 = q0 + dq
        set_q(q0)
        r.update_scene(data, scene_option=opt, camera=cam)
        frames.append(r.render())
media.show_video(frames, fps=fps)

ee_final, _ = fk_analytic(*q0)
print(f"EE drift during null‑space motion: {np.linalg.norm(ee_final - ee0):.2e}")

---

## Summary

| Concept                             | Notation            | Demonstrated                                      |
|-------------------------------------|---------------------|---------------------------------------------------|
| Differential motion                 | $D = [dp;\; \delta]$ | Skew‑symmetric matrix $\hat{z}$                   |
| Geometric Jacobian (column‑by‑col)  | $J_i = [z\times r; z]$ | Built from `jnt_axis`, `xanchor`; vs `mj_jac`    |
| Analytical Jacobian                 | $\partial f / \partial q$ | Direct differentiation of FK                  |
| RMRC (resolved motion rate control) | $\dot{q} = J^{+} \dot{x}_d$ | Cartesian path following animation            |
| Singularity / manipulability        | $w = \sqrt{\det(J J^T)}$ | Interactive ellipsoid, workspace heatmap       |
| Static forces                       | $\tau = J^T F$      | Verified with `mj_inverse`                        |
| Redundancy / null space             | $\dot{q}_0 \in \mathcal{N}(J)$ | Null‑space projector, self‑motion animation |